# `financialtools.utils` — User Guide

I/O helpers, ticker/sector lookups, financial-data loaders, and batch-file utilities.

```
financialtools/utils.py
│
├── I/O Helpers
│   ├── export_to_csv(df, path)                     → None
│   └── export_to_xlsx(df, path, sheet_name)        → None
│
├── Ticker / Sector Lookups
│   ├── get_tickers(filepath, columns, pattern)      → pl.DataFrame | pl.Series
│   ├── get_sector_for_ticker(ticker)                → str
│   ├── get_market_metrics(sector, year, file_path)  → pd.DataFrame
│   └── list_evaluated_tickers(base_dir)             → list[str]
│
├── Weight Utilities
│   └── flatten_weights(weights)                     → dict
│
├── Data Conversion
│   └── dataframe_to_json(df)                        → str
│
├── Yahoo Finance Enrichment
│   ├── get_ticker_profile(ticker)                   → pd.DataFrame (1 row)
│   └── enrich_tickers(df, ticker_column)            → pd.DataFrame
│
├── Financial Results Loader
│   ├── get_fin_data(ticker, year, base_dir, round_metrics) → tuple[str, str, str]
│   └── get_fin_data_year(ticker, year)              → tuple[str, str, str]  (alias)
│
└── Batch File Utilities
    └── create_newbatch_folder(file_name, batch_job_id) → None
```

| Function | Needs files on disk | Needs internet | Raises |
|---|---|---|---|
| `export_to_csv` | no | no | — (errors printed) |
| `export_to_xlsx` | no | no | — (errors printed) |
| `get_tickers` | `sector_ticker.txt` | no | `FileNotFoundError` |
| `dataframe_to_json` | no | no | `TypeError` |
| `get_sector_for_ticker` | `sector_ticker.txt` | no | `SectorNotFoundError` |
| `get_market_metrics` | `metrics_by_sectors.xlsx` | no | `SectorNotFoundError` |
| `list_evaluated_tickers` | `financial_data/composite_scores.xlsx` | no | — (returns `[]` on failure) |
| `flatten_weights` | no | no | — (errors printed) |
| `get_ticker_profile` | no | **yes** (yfinance) | `ImportError` |
| `enrich_tickers` | no | **yes** (yfinance) | `ValueError` if all fail |
| `get_fin_data` | `financial_data/*.xlsx` | no | `FileNotFoundError` |
| `get_fin_data_year` | `financial_data/*.xlsx` | no | `FileNotFoundError` |
| `create_newbatch_folder` | input file must exist | no | `FileNotFoundError` |

> **Sector weights** — use `financialtools.config.sector_metric_weights` directly.
> `get_sector_weights()` has been removed. Build the weights DataFrame with:
> ```python
> from financialtools.config import sector_metric_weights
> import pandas as pd
> weights = (pd.DataFrame(list(sector_metric_weights["Technology"].items()),
>                         columns=["metrics", "weights"]).assign(sector="Technology"))
> ```

**Prerequisites:**
- Sections 1–6 (offline demos) run with **no API key and no network**.
- Section 7 (`list_evaluated_tickers`) requires `financial_data/composite_scores.xlsx` from the evaluation pipeline.
- Section 8 (`get_market_metrics`, `get_fin_data`) requires all runtime data files from the evaluation pipeline.
- Section 9 (`get_ticker_profile`, `enrich_tickers`) requires internet access and `yfinance`.

## Sections
1. [Setup](#1--setup)
2. [Sample data](#2--sample-data)
3. [export_to_csv / export_to_xlsx](#3--export_to_csv--export_to_xlsx)
4. [flatten_weights](#4--flatten_weights)
5. [dataframe_to_json](#5--dataframe_to_json)
6. [get_tickers (mock demo)](#6--get_tickers)
7. [Live: list_evaluated_tickers](#7--live-list_evaluated_tickers)
8. [Live: get_market_metrics / get_fin_data](#8--live-sector--financial-data-functions)
9. [Live: get_ticker_profile / enrich_tickers](#9--live-get_ticker_profile--enrich_tickers)
10. [Error handling reference](#10--error-handling-reference)
11. [Common failure modes](#11--common-failure-modes)
12. [End-to-end pattern](#12--end-to-end-pattern)

## 1 — Setup

In [ ]:
import json
import os
import tempfile

import pandas as pd
import polars as pl

from financialtools.utils import (
    export_to_csv,
    export_to_xlsx,
    get_tickers,
    dataframe_to_json,
    flatten_weights,
    get_sector_for_ticker,
    get_market_metrics,
    list_evaluated_tickers,
    get_ticker_profile,
    enrich_tickers,
    get_fin_data,
    get_fin_data_year,
    create_newbatch_folder,
)
from financialtools.exceptions import SectorNotFoundError

print("Imports OK")

## 2 — Sample data

A small in-memory DataFrame reused across Sections 3–5. No files on disk required.

In [ ]:
SAMPLE_DF = pd.DataFrame({
    "ticker":  ["AAPL", "MSFT", "GOOGL"],
    "sector":  ["Technology", "Technology", "Technology"],
    "score":   [4.2, 3.8, 4.0],
    "year":    [2023, 2023, 2023],
})
print(SAMPLE_DF)

## 3 — `export_to_csv` / `export_to_xlsx`

Both functions are thin pandas I/O wrappers.

**Signatures:**
```python
export_to_csv(df: pd.DataFrame, path: str) -> None
export_to_xlsx(df: pd.DataFrame, path: str, sheet_name: str) -> None
```

Key behaviours:
- `export_to_csv` writes without a row index (`index=False`).
- `export_to_xlsx` uses `openpyxl` as the engine; the sheet is named by `sheet_name`.
- Both functions **print** a confirmation on success. Exceptions are caught internally and printed rather than re-raised — check that the output file actually exists when correctness matters.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    # --- CSV round-trip ---
    csv_path = os.path.join(tmp, "scores.csv")
    export_to_csv(SAMPLE_DF, csv_path)
    csv_back = pd.read_csv(csv_path)
    print(f"CSV round-trip OK : {csv_back.shape} rows×cols")
    print(csv_back.to_string(index=False))

    print()

    # --- Excel round-trip ---
    xlsx_path = os.path.join(tmp, "scores.xlsx")
    export_to_xlsx(SAMPLE_DF, xlsx_path, sheet_name="results")
    xlsx_back = pd.read_excel(xlsx_path, sheet_name="results")
    print(f"XLSX round-trip OK: {xlsx_back.shape} rows×cols")
    print(xlsx_back.to_string(index=False))

## 4 — `flatten_weights`

**Signature:**
```python
flatten_weights(weights: dict) -> dict
```

`config.py` stores scoring weights in a *grouped* format where top-level keys are category names and values are `{metric: weight}` dicts. `flatten_weights` collapses that one level of nesting into a single `{metric: weight}` dict that scoring functions expect.

Rules:
- If a value is a `dict` it is unpacked (grouped schema).
- If a value is a scalar it is passed through as-is (already-flat schema).
- Mixed inputs work: flat entries and grouped entries can coexist.
- On any exception the function prints the error and returns `{}` rather than raising.

In [ ]:
# --- Grouped input (the common case from config.py) ---
grouped = {
    "Profitability & Margins": {"GrossMargin": 10, "OperatingMargin": 12},
    "Return":                  {"ROA": 10, "ROE": 12},
    "Leverage":                {"DebtToEquity": 8},
}
flat = flatten_weights(grouped)
print("Grouped -> flat:")
for k, v in flat.items():
    print(f"  {k:<20} : {v}")

print()

# --- Already-flat input passes through unchanged ---
already_flat = {"GrossMargin": 10, "ROA": 6, "DebtToEquity": 8}
print("Already flat -> flat:", flatten_weights(already_flat))

print()

# --- Mixed input ---
mixed = {
    "Profitability": {"GrossMargin": 10},   # grouped
    "FCFYield": 8,                           # flat
}
print("Mixed -> flat:", flatten_weights(mixed))

## 5 — `dataframe_to_json`

**Signature:**
```python
dataframe_to_json(df: pd.DataFrame) -> str
```

Serialises a pandas DataFrame to a JSON string in `records` orientation (list of row dicts). The result can be embedded directly in an LLM prompt.

Raises `TypeError` if the input is not a `pd.DataFrame`.

In [ ]:
# --- Happy path ---
json_str = dataframe_to_json(SAMPLE_DF)
print("Type  :", type(json_str))
print("Output:", json_str)

print()

# --- Verify it is valid JSON ---
parsed = json.loads(json_str)
print(f"Parsed : {len(parsed)} record(s), first = {parsed[0]}")

## 6 — `get_tickers`

**Signature:**
```python
get_tickers(
    filepath: str = "financialtools/data/sector_ticker.txt",
    columns: list | str | None = None,
    pattern: str | None = None,
) -> pl.DataFrame | pl.Series
```

Reads a tab-separated file with columns `ticker`, `sector`, `name`, `marginabile`.

Return-type rules:

| `columns` value | Return type |
|---|---|
| `None` | `pl.DataFrame` — all four columns |
| `"ticker"` (str) | `pl.Series` |
| `["ticker"]` (single-item list) | `pl.Series` |
| `["ticker", "sector"]` (multi-item list) | `pl.DataFrame` with only those columns |

When `pattern` is given, rows are pre-filtered by a regex match on the `ticker` column (via Polars `str.contains`).

The cell below builds an in-memory mock so the demo runs without the real data file.

In [ ]:
# Mock get_tickers: write a temp TSV and call the real function against it.
import tempfile, os
import polars as pl

MOCK_TSV = "\t".join(["ticker", "sector", "name", "marginabile"]) + "\n"
MOCK_TSV += "\n".join([
    "AAPL\tTechnology\tApple Inc\t1",
    "MSFT\tTechnology\tMicrosoft Corp\t1",
    "ENI.MI\tEnergy\tEni SpA\t1",
    "TIT.MI\tTelecommunication\tTelecom Italia\t0",
])

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as f:
    f.write(MOCK_TSV)
    mock_path = f.name

try:
    # All columns (DataFrame)
    all_df = get_tickers(filepath=mock_path)
    print(f"All columns  : type={type(all_df).__name__}, shape={all_df.shape}")
    print(all_df)
    print()

    # Single column as string -> Series
    ticker_series = get_tickers(filepath=mock_path, columns="ticker")
    print(f"columns='ticker' : type={type(ticker_series).__name__}")
    print(ticker_series.to_list())
    print()

    # Multi-column subset -> DataFrame
    subset_df = get_tickers(filepath=mock_path, columns=["ticker", "sector"])
    print(f"columns=['ticker','sector'] : type={type(subset_df).__name__}")
    print(subset_df)
    print()

    # Pattern filter
    mi_tickers = get_tickers(filepath=mock_path, columns="ticker", pattern=r"\.MI")
    print(f"pattern='.MI' tickers : {mi_tickers.to_list()}")

finally:
    os.unlink(mock_path)

## 7 — Live: `list_evaluated_tickers`

> **LIVE cell** — requires `financial_data/composite_scores.xlsx` to exist.
> Run the evaluation pipeline first (`python scripts/run_pipeline.py`).

**Signature:**
```python
list_evaluated_tickers(base_dir: str = "financial_data") -> list[str]
```

Returns a sorted list of ticker symbols that appear in `composite_scores.xlsx`.
This is the canonical way to discover which tickers are ready for analysis — used
internally by `tools.list_available_tickers()`.

Key behaviours:
- Returns `[]` (empty list) on any I/O or parsing failure — never raises.
- Deduplicates and sorts alphabetically.
- `base_dir` defaults to `"financial_data"` (the runtime output directory).

Use this to gate subsequent calls: check the list before calling `get_fin_data`
or `get_stock_evaluation_report` to avoid `FileNotFoundError` on missing tickers.

In [ ]:
# LIVE — requires financial_data/composite_scores.xlsx

# --- Happy path: list all evaluated tickers ---
tickers = list_evaluated_tickers()
print(f"Evaluated tickers ({len(tickers)} total):")
print(tickers[:20], "..." if len(tickers) > 20 else "")

print()

# --- Custom base_dir ---
# If your pipeline wrote to a different directory:
# tickers_alt = list_evaluated_tickers(base_dir="my_output_dir")

# --- Graceful failure: missing directory returns [] ---
empty = list_evaluated_tickers(base_dir="/nonexistent/path")
print(f"Missing dir returns: {empty!r}")  # []

print()

# --- Guard pattern: check before calling get_fin_data ---
ticker_to_check = "AAPL"
if ticker_to_check in list_evaluated_tickers():
    metrics, composite, red_flags = get_fin_data(ticker_to_check, year=2023)
    print(f"{ticker_to_check}: data loaded OK")
else:
    print(f"{ticker_to_check}: not yet evaluated — run the pipeline first")